# AQI Forecasting — Prophet — India

Trains a Prophet time-series model per region to forecast AQI (PM2.5-based)
using **real historical data** from Open-Meteo's Air Quality API (CAMS
reanalysis) — free, no key needed.

Produces one model per region (Nainital, Delhi) and saves them for the
backend to load.

In [ ]:
!pip install -q prophet pandas numpy requests joblib matplotlib

In [ ]:
import requests
import pandas as pd
from datetime import date, timedelta

REGIONS = {
    "uk-nainital": {"lat": 29.3803, "lon": 79.4636},
    "dl-delhi": {"lat": 28.6139, "lon": 77.2090},
}

HISTORY_DAYS = 365  # Open-Meteo air quality archive covers well over a year

## 1. Download real historical daily-average PM2.5 per region

In [ ]:
def fetch_historical_pm25(lat, lon, start, end):
    resp = requests.get(
        "https://air-quality-api.open-meteo.com/v1/air-quality",
        params={
            "latitude": lat, "longitude": lon,
            "hourly": "pm2_5",
            "start_date": start, "end_date": end,
            "timezone": "Asia/Kolkata",
        },
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()["hourly"]
    df = pd.DataFrame({"ds": pd.to_datetime(data["time"]), "pm2_5": data["pm2_5"]})
    # Prophet works on daily granularity here — average the hourly readings per day
    daily = df.set_index("ds").resample("D")["pm2_5"].mean().reset_index()
    return daily.dropna()

end = date.today().isoformat()
start = (date.today() - timedelta(days=HISTORY_DAYS)).isoformat()

region_data = {}
for region_id, coords in REGIONS.items():
    df = fetch_historical_pm25(coords["lat"], coords["lon"], start, end)
    region_data[region_id] = df
    print(f"{region_id}: {len(df)} days of real PM2.5 data")

region_data["dl-delhi"].tail()

## 2. Train one Prophet model per region

In [ ]:
from prophet import Prophet
import joblib

models = {}
for region_id, df in region_data.items():
    df = df.rename(columns={"pm2_5": "y"})
    m = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
    m.fit(df)
    models[region_id] = m
    print(f"Trained model for {region_id}")

## 3. Forecast next 7 days + evaluate

In [ ]:
for region_id, m in models.items():
    future = m.make_future_dataframe(periods=7)
    forecast = m.predict(future)
    print(f"\n{region_id} — next 7 days forecast:")
    print(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(7).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(models), 1, figsize=(10, 4 * len(models)))
for ax, (region_id, m) in zip(axes, models.items()):
    future = m.make_future_dataframe(periods=7)
    forecast = m.predict(future)
    m.plot(forecast, ax=ax)
    ax.set_title(f"AQI (PM2.5) forecast — {region_id}")
plt.tight_layout()
plt.show()

## 4. Save the trained models

In [ ]:
for region_id, m in models.items():
    filename = f"aqi_forecast_{region_id}.pkl"
    joblib.dump(m, filename)
    print(f"Saved {filename}")

from google.colab import files
import glob
for f in glob.glob("aqi_forecast_*.pkl"):
    files.download(f)